In [1]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
from dotenv import load_dotenv
import os

C:\Users\suhee\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv('.env')
kkey = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=kkey)

In [ ]:
index = pc.create_index(
    name= os.getenv('PINECONE_INDEX_NAME'),
    dimension=768,
    metric="cosine",
    spec=ServerlessSpec(
        cloud = 'aws',
        region= os.getenv("PINECONE_ENVIRONMENT")
    ))
# the above code CREATES a new index, which we obv dont need!

In [4]:
pc.list_indexes()

[
    {
        "name": "suwu-rag",
        "metric": "cosine",
        "host": "suwu-rag-r3dfh16.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 768,
        "deletion_protection": "disabled",
        "tags": null
    }
]

In [5]:
index = pc.Index(os.getenv('PINECONE_INDEX_NAME'))

In [6]:
from pdfminer.high_level import extract_text

text = extract_text('Syllabus.pdf')

In [7]:
import re
import unicodedata

def clean_pdf_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r", "")
    text = re.sub(r"-\n(\s*)", "", text)
    text = re.sub(r"(?<=[a-zA-Z0-9]),?\n(?=[a-zA-Z0-9])", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = re.sub(r"([,.!?;:])(?=[A-Za-z0-9])", r"\1 ", text)
    text = "\n".join(line.strip() for line in text.split("\n"))
    return text.strip()


In [8]:
def chunk_text(text: str, chunk_size=1000, overlap=200):
    """
    Splits text into overlapping chunks.
    chunk_size and overlap are character-based approximations.
    """
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start = start + chunk_size - overlap

    return chunks


In [9]:
text = clean_pdf_text(text)
data = chunk_text(text)

In [10]:
len(data)

224

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

emb = HuggingFaceEmbeddings(model="sentence-transformers/all-mpnet-base-v2")
result = emb.embed_documents(texts=data)

In [13]:
vectors_to_upsert = []
for i, vector in enumerate(result):
    vectors_to_upsert.append({
        "id": f"chunk-{i}",
        "values": vector,
        "metadata": {
            "text": data[i]
        }
    })


In [14]:
index = pc.Index(host=os.getenv("HOST_KEY"))
index.upsert(
    namespace= "mysyllabus",
    vectors= vectors_to_upsert
)
print("done!")

done!
